# 02 · Честная оценка идентификации и re-ID
**Результат:** реализовать L2-нормировку, FAR/TAR и калибровку, оценку retrieval с исключениями, обучить простую коррекцию камеры на train и сравнить с baseline на фиксированном test.

План: 120–160 минут работы; вычисления — секунды/минуты на CPU. Готовую сеть не обучаем. Пять TODO намеренно останавливают «Запустить все», пока вы не реализуете решение. Код демо разрешен, но объяснение и собственные контрпримеры обязательны.

**Почему синтетика здесь обоснована.** Наблюдение = identity + camera offset + noise. Мы знаем, какой фактор изменен, можем разделить ID и проверить, не выдает ли легкий кадр той же камеры ложный успех. В реальных embeddings эти причины смешаны и вычитание среднего не обязано помогать. Это эксперимент о протоколе и доменном сдвиге, не результат на Market-1501, не измерение качества OSNet и не способ идентифицировать реальных людей.

Нужны `numpy matplotlib`; JupyterLab/Colab. Ничего не загружается. Не добавляйте результаты, данные и выполненное решение в общий репозиторий.

In [ ]:
# Запускайте первой. В Colab сохраняем штатные PyTorch/CUDA и ipywidgets.
import importlib.util, subprocess, sys
if sys.version_info < (3, 10):
    raise RuntimeError('Нужен Python 3.10 или новее.')
_packages = {'numpy': 'numpy>=1.26,<3', 'matplotlib': 'matplotlib>=3.8,<4'}
_missing = [spec for module, spec in _packages.items()
            if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])
    raise RuntimeError('Пакеты установлены. Перезапустите среду выполнения и выполните ноутбук сверху.')
import numpy, matplotlib
if not (1 <= int(numpy.__version__.split('.')[0]) < 3):
    raise RuntimeError('Нужен NumPy 1.x или 2.x; используйте отдельное окружение.')
if int(matplotlib.__version__.split('.')[0]) != 3:
    raise RuntimeError('Нужен Matplotlib 3.x; используйте отдельное окружение.')
_versions = {'python': sys.version.split()[0], 'numpy': numpy.__version__,
             'matplotlib': matplotlib.__version__}
if 'scipy' in _packages:
    import scipy
    if int(scipy.__version__.split('.')[0]) != 1:
        raise RuntimeError('Нужен SciPy 1.x; используйте отдельное окружение.')
    _versions['scipy'] = scipy.__version__
if 'cv2' in _packages:
    import cv2
    _versions['opencv'] = cv2.__version__
    _required_features = ()
    _absent = [name for name in _required_features if not hasattr(cv2, name)]
    if _absent:
        raise RuntimeError('В OpenCV нет ' + ', '.join(_absent) +
                           '. Установите один вариант OpenCV 4.12.0.88 в отдельной среде и перезапустите ядро.')
if 'ipywidgets' in _packages:
    import ipywidgets
    _versions['ipywidgets'] = ipywidgets.__version__
    if int(ipywidgets.__version__.split('.')[0]) not in (7, 8):
        raise RuntimeError('Нужен ipywidgets 7.x или 8.x; в Colab 2026.07 оставьте штатный 7.7.1.')
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except ImportError:
        pass
print('Окружение:', _versions)
print('Вычисления этой лабораторной: CPU; GPU не требуется.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import time, json, csv, hashlib
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({'figure.dpi':110, 'font.size':11})

## До запуска: гипотеза и протокол
Запишите предсказание: как camera subtraction повлияет на mAP, rank-1 и TAR при ограничении FAR? Почему эффект может быть разным? Назовите хотя бы одну причину, по которой этот прием провалится на реальной камере.

**Заполните:** гипотеза, причина, ожидаемый график, условие опровержения.

Train: 24 ID, val: 20 других ID, test: еще 20 ID. Три одинаковые камеры доступны во всех split; это перенос на **новые ID в известных камерах**, не обобщение на неизвестную камеру. На каждого ID — два наблюдения каждой камеры. Сдвиги камер оцениваются только на train. Пять alpha выбираются по val mAP; ties → меньшая alpha. Порог каждого сравниваемого варианта калибруется по val: max TAR при FAR≤.05; ties → больший threshold. До test фиксируем alpha и пороги.

Нельзя использовать ID val/test для оценки camera offsets или менять шум, seeds и split после просмотра результата.

## TODO 1 · Направление вектора
Реализуйте нормировку без изменения входа. Нулевой вектор отвергается, чтобы отсутствие признака не маскировалось под корректное сходство.

In [ ]:
def normalize_rows(x):
    """Finite matrix [N,D] -> unit rows. Reject nonfinite/zero rows (norm<=1e-12). Empty N is valid."""
    raise NotImplementedError('TODO 1: L2-normalization')

In [ ]:
assert np.allclose(normalize_rows([[3,4],[0,2]]),[[.6,.8],[0,1]])
assert normalize_rows(np.empty((0,3))).shape==(0,3)
for bad in ([[0,0]],[[np.nan,1]],[1,2]):
    try:normalize_rows(bad)
    except ValueError:pass
    else:raise AssertionError('must reject invalid vector')
print('TODO 1: open tests OK')

## TODO 2 · Калибровка verification
Напишите два разных знаменателя и правило score≥threshold. Используйте конечный набор порогов, включая вариант «не принять никого». Порог подбирается по val, а не по test.

In [ ]:
def pair_rates(scores, same, threshold):
    """1D finite scores, boolean same; both classes required. Accept score>=threshold.
    Return dict FAR/TAR; reject shape mismatch, NaN threshold, missing class."""
    raise NotImplementedError('TODO 2a: FAR and TAR')

def choose_threshold(scores,same,max_far):
    """Candidates: unique scores plus nextafter(max score,+inf). Target in [0,1].
    Among FAR<=max_far maximize TAR; for equal TAR choose LARGEST threshold.
    All candidate rates use pair_rates. Return float threshold."""
    raise NotImplementedError('TODO 2b: validation calibration')

In [ ]:
s=np.array([.9,.8,.7,.4]);y=np.array([True,False,True,False])
assert pair_rates(s,y,.8)=={'FAR':.5,'TAR':.5}
assert choose_threshold(s,y,0)==.9
assert choose_threshold(s,y,.5)==.7
assert choose_threshold([.5,.5],[True,False],0)>.5
assert pair_rates([.5,.5],[True,False],.5)=={'FAR':1.,'TAR':1.}
for bad in [([.5],[True]),([.5,np.nan],[True,False])]:
    try:pair_rates(*bad,.5)
    except ValueError:pass
    else:raise AssertionError('must reject invalid pair set')
print('TODO 2: open tests OK')

## TODO 3 · Полный оценщик query/gallery
AP — средняя precision на всех релевантных позициях после исключений. Не усредняйте только успешные запросы. Запрос без допустимых positives исключается из среднего с явным счетчиком; это свойство протокола, а не нулевая AP.

In [ ]:
def retrieval_metrics(scores,q_ids,q_cams,g_ids,g_cams):
    """Validate shapes/finite scores. Exclude same ID AND same camera.
    Sort scores descending; stable ties. Skip queries without eligible positives.
    Return mAP,rank1,valid_queries,skipped_queries,per_query.
    per_query has dict query,AP,rank1,top_gallery for each valid query.
    If all skipped, mAP/rank1 are NaN. Empty gallery is allowed."""
    raise NotImplementedError('TODO 3: full retrieval evaluation')

In [ ]:
r=retrieval_metrics([[.99,.9,.8,.7]],[1],[0],[1,2,1,1],[0,1,1,2])
assert r['rank1']==0 and np.isclose(r['mAP'],(1/2+2/3)/2)
# Stable ties: first negative then positive.
r=retrieval_metrics([[.8,.8]],[1],[0],[2,1],[1,1]);assert r['mAP']==.5
r=retrieval_metrics([[.8]],[1],[0],[1],[0]);assert r['skipped_queries']==1 and np.isnan(r['mAP'])
r=retrieval_metrics(np.empty((1,0)),[1],[0],[],[]);assert r['valid_queries']==0
print('TODO 3: open tests OK')

## TODO 4 · Простая обучаемая коррекция
Оцените сдвиг камеры как camera mean − global mean. Затем вычитайте alpha·shift из сырых признаков и только после этого выполняйте L2-нормировку. При сбалансированных ID среднее личности одинаково для камер. При несбалансированных камерах это предположение нарушается.

In [ ]:
def fit_camera_offsets(x,cams):
    """TRAIN rows only; dict camera:int -> camera_mean - global_mean.
    Validate finite 2D nonempty data and camera vector shape."""
    raise NotImplementedError('TODO 4a: estimate camera shift')

def apply_camera_offsets(x,cams,offsets,alpha):
    """Subtract alpha*offset then normalize_rows. alpha in [0,1].
    Reject wrong shapes and unknown camera. Do not mutate input."""
    raise NotImplementedError('TODO 4b: apply frozen transformation')

In [ ]:
x=np.array([[1.,1.],[3,1],[1,3],[3,3]]);c=np.array([0,0,1,1]);before=x.copy()
o=fit_camera_offsets(x,c);assert np.allclose(o[0],[0,-1]) and np.allclose(o[1],[0,1])
assert np.allclose(apply_camera_offsets(x,c,o,0),normalize_rows(x));assert np.array_equal(x,before)
assert np.allclose(apply_camera_offsets(x,c,o,1)[0],normalize_rows([[1,2]])[0])
try:apply_camera_offsets([[1.,2.]],[9],o,.5)
except ValueError:pass
else:raise AssertionError('unseen camera must not silently pass')
print('TODO 4: open tests OK')

## TODO 5 · Собственные проверки с прогнозом
Создайте два новых контрольных случая, укажите ожидаемые числа до запуска и объясните, какую типичную неправильную реализацию они выявляют. Не копируйте открытые тесты.

In [ ]:
# TODO 5. До кода запишите ожидаемый ответ для нового контрпримера.
# Пример должен проверять исключение камеры, ties или отсутствие positives
# и отличаться от всех открытых тестов. Второй — проверять нулевой вектор
# либо отсутствующий класс пары. Минимум две assert-проверки.
raise NotImplementedError('TODO 5: your predicted counterexamples')

## Контроль данных и baseline
Проверьте разбиение по ID, число кадров/камер и одинаковый бюджет данных. Каждая пятая пара identity близка: «двойник» может остаться ошибкой даже после идеального устранения камеры. Открывайте test только после таблицы val и записи выбора.

In [ ]:
# Генератор наблюдений: identity + общий camera offset + Gaussian noise.
# «Двойники» имеют близкие identity-векторы; seed и диапазоны ID фиксированы.
CAMERA_SEED=730; D=16
camera_rng=np.random.default_rng(CAMERA_SEED)
TRUE_CAMERA=camera_rng.normal(size=(3,D));TRUE_CAMERA-=TRUE_CAMERA.mean(0)
TRUE_CAMERA*=.95

def make_split(seed,start_id,n_id):
    rng=np.random.default_rng(seed)
    identity=rng.normal(size=(n_id,D))*.58
    # Каждая пятая пара ID похожа по внешности; это не биометрическая модель.
    for j in range(1,n_id,5):identity[j]=identity[j-1]+rng.normal(0,.17,D)
    x=[];ids=[];cams=[];rep=[]
    for j in range(n_id):
        for c in range(3):
            for k in range(2):
                x.append(identity[j]+TRUE_CAMERA[c]+rng.normal(0,.42,D))
                ids.append(start_id+j);cams.append(c);rep.append(k)
    return {'x':np.array(x),'id':np.array(ids),'cam':np.array(cams),'rep':np.array(rep)}

train=make_split(101,0,24);val=make_split(202,100,20)
assert set(train['id']).isdisjoint(set(val['id']))
# В каждом split все ID имеют одинаковое число кадров каждой камеры.
for d in [train,val]:
    assert all(np.sum((d['id']==i)&(d['cam']==c))==2 for i in np.unique(d['id']) for c in range(3))
print('train:',len(train['x']),'rows / 24 IDs; val:',len(val['x']),'rows / 20 IDs')

## Контролируемый опыт на val
Меняется **только alpha** (0, .25, .5, .75, 1). Raw baseline = alpha 0. Формула и offsets одинаковы. Query — первое наблюдение камеры 0; gallery — все остальные наблюдения. Retrieval оставляет negatives той же камеры; same-ID/same-camera исключаются.

Для verification здесь отдельно заданы **все cross-camera query/gallery пары**. Укажите число negative-пар и шаг FAR. Пары с общим query зависимы: нельзя считать их независимыми людьми и строить наивные доверительные интервалы.

Сначала изучите вывод val, заполните объяснение выбора, затем выполните ячейку фиксации/оценки. Автовыбор обеспечивает воспроизводимость; ваше обоснование должно ссылаться на измерение.

In [ ]:
RESULTS=Path('results');RESULTS.mkdir(exist_ok=True)
FAR_TARGET=.05
ALPHAS=[0.,.25,.5,.75,1.]
offsets=fit_camera_offsets(train['x'],train['cam'])

def score_split(data,alpha):
    features=apply_camera_offsets(data['x'],data['cam'],offsets,alpha)
    # Query: одна запись camera0; gallery: все оставшиеся записи.
    qi=np.flatnonzero((data['cam']==0)&(data['rep']==0))
    gi=np.flatnonzero(~((data['cam']==0)&(data['rep']==0)))
    scores=features[qi]@features[gi].T
    ranking=retrieval_metrics(scores,data['id'][qi],data['cam'][qi],data['id'][gi],data['cam'][gi])
    # Verification: только cross-camera пары query-gallery; negatives той же камеры
    # исключены здесь по определению парного протокола, но остаются в retrieval.
    pair_mask=data['cam'][qi,None]!=data['cam'][gi][None,:]
    same=data['id'][qi,None]==data['id'][gi][None,:]
    return scores,ranking,scores[pair_mask],same[pair_mask],qi,gi

def evaluate(data,alpha,threshold=None):
    start=time.perf_counter();scores,r,ps,same,qi,gi=score_split(data,alpha)
    threshold=choose_threshold(ps,same,FAR_TARGET) if threshold is None else threshold
    rates=pair_rates(ps,same,threshold)
    row={'alpha':alpha,'threshold':threshold,'mAP':r['mAP'],'rank1':r['rank1'],**rates,'valid_queries':r['valid_queries'],'negative_pairs':int((~same).sum()),'seconds':time.perf_counter()-start}
    return row,(scores,r,qi,gi)

val_rows=[evaluate(val,a)[0] for a in ALPHAS]
# Только mAP определяет alpha; при точном равенстве выбирается меньшая alpha.
selected=max(val_rows,key=lambda r:(r['mAP'],-r['alpha']))
baseline=val_rows[0]
locked={'selected_alpha':selected['alpha'],'selected_threshold':selected['threshold'],'baseline_threshold':baseline['threshold'],'FAR_TARGET':FAR_TARGET,'selection':'max val mAP; ties smaller alpha','train_seed':101,'val_seed':202,'test_seed':303,'camera_seed':730}
(RESULTS/'selection_locked.json').write_text(json.dumps(locked,indent=2),encoding='utf8')
with (RESULTS/'validation.csv').open('w',newline='') as f:
    writer=csv.DictWriter(f,fieldnames=val_rows[0]);writer.writeheader();writer.writerows(val_rows)
for row in val_rows:print({k:round(v,4) if isinstance(v,float) else v for k,v in row.items()})
print('LOCKED:',locked)
fig,axs=plt.subplots(1,2,figsize=(9,3))
axs[0].plot(ALPHAS,[r['mAP'] for r in val_rows],'o-',label='mAP');axs[0].plot(ALPHAS,[r['rank1'] for r in val_rows],'s-',label='rank-1');axs[0].legend()
axs[1].plot(ALPHAS,[r['TAR'] for r in val_rows],'o-',label='val TAR at calibrated FAR');axs[1].legend()
for ax in axs:ax.set(xlabel='alpha (camera subtraction)',ylim=(0,1));ax.grid()
fig.tight_layout();fig.savefig(RESULTS/'validation.png');plt.show();plt.close(fig)

## Решение до test
**Заполните:** выбранная alpha, val mAP/rank-1, пороги baseline/selected, FAR/TAR и почему это не доказательство работы на неизвестной камере. Назовите вероятную оставшуюся ошибку. Сохраненный `selection_locked.json` задает фиксированный выбор. Ниже test оценивается один раз для baseline и выбранной настройки.

In [ ]:
# Не редактируйте alpha/threshold после открытия test. Повторный поиск требует
# нового test; одинаковый seed обеспечивает воспроизводимость, а не новый test.
assert (RESULTS/'selection_locked.json').exists()
lock=json.loads((RESULTS/'selection_locked.json').read_text())
test=make_split(303,200,20)
assert set(test['id']).isdisjoint(set(train['id'])) and set(test['id']).isdisjoint(set(val['id']))
test_rows=[];details={}
for name,a,tau in [('baseline',0.,lock['baseline_threshold']),('selected',lock['selected_alpha'],lock['selected_threshold'])]:
    row,detail=evaluate(test,a,tau);row={'method':name,**row};test_rows.append(row);details[name]=detail
with (RESULTS/'test.csv').open('w',newline='') as f:
    writer=csv.DictWriter(f,fieldnames=test_rows[0]);writer.writeheader();writer.writerows(test_rows)
print('FIXED TEST:')
for row in test_rows:print({k:round(v,4) if isinstance(v,float) else v for k,v in row.items()})
# Одинаковые запросы обоих методов: первые шесть ID, без cherry-picking.
fig,axes=plt.subplots(2,1,figsize=(11,6),constrained_layout=True)
for ax,name in zip(axes,['baseline','selected']):
    scores,r,qi,gi=details[name]
    matrix=scores[:6].copy()
    invalid=(test['id'][qi[:6],None]==test['id'][gi][None,:])&(test['cam'][qi[:6],None]==test['cam'][gi][None,:])
    matrix[invalid]=np.nan
    im=ax.imshow(matrix,vmin=-1,vmax=1,cmap='coolwarm',aspect='auto')
    for j in range(6):
        positives=np.flatnonzero((test['id'][gi]==test['id'][qi[j]])&~invalid[j])
        ax.scatter(positives,np.full(len(positives),j),facecolors='none',edgecolors='black',s=36)
    ax.set(title=name+' | circles: eligible positives; white: excluded',ylabel='Same six queries',xlabel='Gallery index')
fig.colorbar(im,ax=axes,label='cosine score',shrink=.8);fig.savefig(RESULTS/'same_queries.png',bbox_inches='tight');plt.show();plt.close(fig)
# Сохраняем диагностические строки всех query обоих методов.
errors=[]
for name,(_,r,qi,gi) in details.items():
    for row in r['per_query']:
        q=qi[row['query']];g=gi[row['top_gallery']]
        errors.append({'method':name,'query_ID':int(test['id'][q]),'top_ID':int(test['id'][g]),'top_camera':int(test['cam'][g]),'AP':row['AP'],'rank1':row['rank1']})
with (RESULTS/'per_query.csv').open('w',newline='') as f:
    writer=csv.DictWriter(f,fieldnames=errors[0]);writer.writeheader();writer.writerows(errors)
print('Ошибки выбранного метода:',[r for r in errors if r['method']=='selected' and not r['rank1']])

## Отчет по результату
1. Приложите `validation.csv`, `test.csv`, `selection_locked.json`, два графика и заполненный notebook.
2. Сопоставьте исходную гипотезу с val/test. Если test FAR>.05, объясните, почему ограничение на val не гарантирует его на test. Не перекалибровывайте порог.
3. Выберите один ID с ошибкой из `per_query.csv` и один успешный ID. Сопоставьте **тот же query и gallery** у baseline/selected: что изменилось в первом месте и AP? Объясните, почему rank-1 и mAP могут меняться по-разному.
4. Найдите минимум два ограничения эксперимента (известные камеры, аддитивный сдвиг, синтетические «двойники», небольшой sample). Как проверить одно из них на реальных данных с правом использования?
5. Предложите отрицательный контроль: перепутать train camera labels или оценить offsets на несбалансированном train. До запуска запишите прогноз. Если выполняете дополнительный опыт, оценивайте только val и пометьте его отдельно от основного выбора.

Полные критерии — в `ASSESSMENT.md`. На защите объясните собственный counterexample и один конкретный неверный протокол.

Источники: [FaceNet](https://arxiv.org/abs/1503.03832), [ArcFace](https://arxiv.org/abs/1801.07698), [Market-1501: официальный протокол](https://zheng-lab-anu.github.io/Project/project_reid.html). Все массивы создаются в notebook; внешние модели и изображения не используются.